# Notebook 2: Bayesian Umbrella Quadrature on Alanine Dipeptide



## **IMPORTANT**
Before you run this notebook, make sure you have downloaded the whole "HRSMC_ML" folder and added it to your own Drive. Then, continue from there and use Google Collab.

In the previous notebook we used **Bayesian Optimization (BO)** to find
where the force is zero, i.e. the minima/maxima of the free energy surface.

In this notebook we use **Bayesian Quadrature (BUQ)** to do something more ambitious: reconstruct the **entire free energy surface** A(phi, psi) by integrating the mean force over both dihedral angles. Since we have a different goal, we will use a different acquisition function.

### What is the difference?

| | Bayesian Optimization | Bayesian Quadrature |
|---|---|---|
| **Goal** | Find minimum of f(x) | Compute integral of f(x) |
| **Surrogate** | GP on \|force(phi)\| | GP on force(phi, psi) directly |
| **Acquisition** | Reduce uncertainty near minimum | Reduce uncertainty of integral |
| **Output** | Best (phi) found close to the extrema | A(phi, psi) |

### What you will learn
- How BUQ places new simulation points to reduce **integral uncertainty** in 2D
- How kernel choice and hyperparameters affect the GP prediction over the force
- How the IVR acquisition function differs from EI/PI/LCB
- How sampling patterns differ between BO (notebook 1) and BUQ (this notebook)

For more information, check out our paper: https://arxiv.org/abs/2601.08783


> **Note:** We use `AdipepFromGrid`, which is essentially the same precomputed data as notebook 1 but now uses both dihedral angles (phi, psi). However, BUQ is of course also able to handle a 1D free energy.

## 0. Imports

In [ ]:
# Core scientific Python stack (many already in Colab, but we ensure versions)
!pip install \
    numpy==1.26.4 \
    pandas==2.3.0 \
    matplotlib==3.10.0 \
    scipy==1.12.0 \
    plotly==6.0.1

# Packages specific to this course / BUQ environment
!pip install \
    cython==3.0.12 \
    emcee==3.1.6 \
    emukit==0.4.11 \
    GPy==1.13.2 \
    paramz==0.9.6 \
    scikit-optimize

In [ ]:
# Insert the "buq" package which is one level up from the current directory

import os
import sys
sys.path.insert(0, os.path.abspath("../"))
import numpy as np
import matplotlib.pyplot as plt
from buq import BQConfig, BayesianQuadratureRunner
from buq.sample_systems import Adipep2DFromGrid

import pathlib as pl
parent_dir = pl.Path(os.path.abspath(""))



## 1. The system: alanine dipeptide (phi, psi)

`AdipepFromGrid` is a built-in example in BUQ and it has the precomputed simulation data. For our intuition, lets first look at the free energy surface and the forces in 2D by plotting this data.

In [ ]:
system = Adipep2DFromGrid()

# Evaluate FES on a dense grid for reference
dat = np.loadtxt(parent_dir / "fes_adipep_phi_psi.dat")
phi_flat = dat[:, 0]
psi_flat = dat[:, 1]
fes_flat = dat[:, 2]
dAdphi_flat = dat[:, 3]
dAdpsi_flat = dat[:, 4]
# Reconstruct the 2D grid
phi_vals = np.unique(phi_flat)
psi_vals = np.unique(psi_flat)
PHI, PSI = np.meshgrid(phi_vals, psi_vals)
FES_ref = fes_flat.reshape(len(phi_vals), len(psi_vals))
dAdphi    = dAdphi_flat.reshape(len(psi_vals), len(phi_vals))
dAdpsi    = dAdpsi_flat.reshape(len(psi_vals), len(phi_vals))
FES_ref -= np.min(FES_ref)


# --- Plot FES + gradients ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# FES
cf0 = axes[0].contourf(PHI, PSI, FES_ref, levels=20, cmap="viridis")
plt.colorbar(cf0, ax=axes[0], label=r"A($\phi, \psi$) (kJ/mol)")
axes[0].set_xlabel(r"$\phi$ (rad)")
axes[0].set_ylabel(r"$\psi$ (rad)")
axes[0].set_title("Free Energy Surface")

# ∂A/∂φ
cf1 = axes[1].contourf(PHI, PSI, dAdphi, levels=20, cmap="coolwarm")
plt.colorbar(cf1, ax=axes[1], label=r"$\partial A / \partial \phi$ (kJ/mol·rad$^{-1}$)")
axes[1].set_xlabel(r"$\phi$ (rad)")
axes[1].set_ylabel(r"$\psi$ (rad)")
axes[1].set_title(r"$\partial A / \partial \phi$")

# ∂A/∂ψ
cf2 = axes[2].contourf(PHI, PSI, dAdpsi, levels=20, cmap="coolwarm")
plt.colorbar(cf2, ax=axes[2], label=r"$\partial A / \partial \psi$ (kJ/mol·rad$^{-1}$)")
axes[2].set_xlabel(r"$\phi$ (rad)")
axes[2].set_ylabel(r"$\psi$ (rad)")
axes[2].set_title(r"$\partial A / \partial \psi$")

plt.tight_layout()
plt.show()

## 2. Configure BUQ

`BQConfig` controls the GP kernel and the BUQ loop.

Key parameters:
- `kernel_type`: choise of the GP kernel, you can choose from: `"Matern12"`, `"Matern32"`, `"Matern52"`, `"RBF"`
- `lengthscale`: You can pick different lengthscales for each dimension (phi and psi), controls how quickly the GP varies
- `noise`: assumed observation noise (this is a white kernel)
- `acq_function`: `"IVR"` (Integral Variance Reduction), `"US"` (Uncertainty Sampling).
- `grid_size_2d`: resolution of the internal (phi, psi) grid, this is up to your choice. In 2D this matters more, since a high resolution grid significantly slows down integral computation. 40x40 should give enough resoluation.
- `use_mini` / `fast_mini`: enables to optimize the computation of the integral.

### TODO: try changing these settings and see what happens!
- `kernel_type`
- `lengthscale`
- `noise`

Hint: you can use the initial setting as a guideline, but feel free to explore!

In [ ]:
# --- Settings: change these! ---
kernel_type = "Matern32"
lengthscale = np.array([0.75, 0.75])
noise       = 1e-6
# --------------------------------

config = BQConfig(
    kernel_type=kernel_type,
    lengthscale=lengthscale,
    noise=noise,
    variance=1.0,
    acq_function="IVR",
    grid_size_2d=(40, 40),
    use_mini=True,
    fast_mini=True,
)

In [ ]:
initial_points = np.array([[-1.507964474, 1.193805208 ],  [0.879645943, -0.879645943]])

runner = BayesianQuadratureRunner(system, config)
runner.initialize(initial_points)
runner.run(n_queries=30)

# Plot initial state
runner.plot_fes(show=True)
runner.plot_acq(show=True, full=True)
runner.plot_derivatives(show=True)

## 3. Initialize the runner

We start with a small set of initial points where the force has already been
evaluated. The runner fits an initial GP to these observations.

### TODO: try different initial point placements
- What happens if all initial points are clustered in one region? What do you think are logical or good initial points?


In [ ]:
# --- Settings: change these! ---
# initial_points = np.array([[-1.507964474, 1.193805208 ],  [0.879645943, -0.879645943]])
initial_points = np.array([[-1.50, -1.49 ],  [0.88, 0.85]])


# --------------------------------

runner = BayesianQuadratureRunner(system, config) #if you want, you can rerun the cell above to have different GP settings of course
runner.initialize(initial_points)

runner.run(n_queries=10) #you can also add some sampling here


# Plot initial state
runner.plot_fes(show=True)
runner.plot_acq(show=True, full=True)
runner.plot_derivatives(show=True)

**Questions:**
- Where does the acquisition function suggest sampling next? Why?
- How does the initial FES estimate compare to the reference?

## 4. The BUQ loop

At each step the runner:
1. Evaluates the IVR acquisition function on the (phi, psi) grid
2. Picks the (phi, psi) that maximally reduces integral variance
3. Queries the gradient at that point (`system.get_force`)
4. Updates the GP posterior and recomputes the 2D FES

The acquisition is a weighted combination:

$$\text{acq}(\phi, \psi) = (1 - w_\text{fes}) \cdot \text{IVR}(\phi, \psi) - w_\text{fes} \cdot \hat{F}(\phi, \psi)$$

- `weight_fes=0`: pure IVR, reduces integral variance uniformly (exploration)
- `weight_fes > 0`: biases sampling toward high free energy regions (exploitation)


> **Note** This might take a while, since adding this weight fes means we need to compute the whole free energy landscape for every sample, and that integral takes time. If it is taking too long, you can reduce the grid points from 40 to 20 in the config, see `grid_size_2d=(40, 40)` .
### TODO: try changing the weights and number of steps

In [ ]:
# --- Settings: change these! ---
n_steps    = 5
weight_fes = 0.8

# --------------------------------

runner = BayesianQuadratureRunner(system, config)
runner.initialize(initial_points)
weight_var = 1 - weight_fes
for i in range(n_steps):
    print(f"\n--- Step {i+1}/{n_steps} ---")
    runner.run_one_query(weight_var = weight_var, weight_fes=weight_fes)
    runner.plot_fes(show=True)
    runner.plot_acq(show=True, full=True)

print("Final FES shape:", runner.current_fes_2d.shape)

**Questions:**
- How do the sampling locations compare to notebook 1 (BO)?
- Does BUQ sample near the force zero crossings, or elsewhere? Do you see a pattern?
- What changes when you increase `weight_fes`?

## 5. Effect of kernel and lengthscale

The kernel controls the GP's assumptions about the smoothness of the force.

- **Small lengthscale**: GP varies quickly, can fit sharp features, but needs more points to cover the domain
- **Large lengthscale**: GP varies slowly, smooth FES, may miss sharp features
- **RBF**: infinitely smooth (this is a strong assumption)
- **Matern12/32/52**: less smooth, more realistic for MD forces, where Matern12 is really for sharp, rugged derivatives, while Matern52 expects a smoother derivative.

In 2D you can also use **anisotropic (ARD) lengthscales**, e.g. `[0.6, 1.2]`, if the force varies at different rates along phi and psi.

### TODO: run each cell and compare the FES reconstructions

In [ ]:
# Helper -- no need to change this
def run_buq(kernel_type, lengthscale, noise=1e-6, n_queries=50,
            weight_acq=1.0, weight_fes=0.0, title=""):
    """Run a full BUQ loop and return the runner."""
    cfg = BQConfig(
        kernel_type=kernel_type,
        lengthscale=lengthscale,
        noise=noise,
        variance=1.0,
        acq_function="IVR",
        grid_size_2d=(40, 40),
        use_mini=True,
        fast_mini=True,
    )
    r = BayesianQuadratureRunner(system, cfg)
    r.initialize(np.array([[-1.507964474, 1.193805208 ],  [0.879645943, -0.879645943]]))
    r.run(n_queries=n_queries, weight_var=weight_acq, weight_fes=weight_fes)
    print(f"\n{title}")
    r.plot_derivatives(show=True)
    r.plot_fes(show=True)
    return r

In [ ]:
r_short   = run_buq("Matern32", lengthscale=[0.1, 0.1], title="Matern32, lengthscale=0.1")

In [ ]:
r_default = run_buq("Matern52", lengthscale=[0.75, 0.75], title="Matern52, lengthscale=0.75")

In [ ]:
r_long    = run_buq("Matern32", lengthscale=[3.0, 3.0], title="Matern32, lengthscale=2.0")

In [ ]:
r_ard     = run_buq("Matern32", lengthscale=[0.06, 1.2], title="Matern32, ARD lengthscale=[0.6, 1.2]")

In [ ]:
r_rbf     = run_buq("RBF",      lengthscale=[0.6, 0.6], title="RBF, lengthscale=0.6")

**Questions:**
- Which kernel/lengthscale gives the best FES reconstruction?
- What happens with a short lengthscale?
- Wath happens with a long lengthscale?
- Does using ARD lengthscales improve the reconstruction? Can you think of examples where it does?
- Why might RBF be a poor choice for MD force data?

## 6. Acquisition functions: IVR vs US

So far we used **IVR** (Integral Variance Reduction), which directly minimizes the variance of the integral estimate.

One of the alternatives that is available is Uncertainty Sampling, US, which samples where GP variance is highest. IVR is the most principled choice for BUQ, but US can be useful for comparison.

### TODO: run each cell and compare sampling patterns

In [ ]:
# Note: to use a different acquisition, change acq_function in BQConfig inside run_buq,
# or copy the helper and modify it:

def run_buq_acq(acq_function, n_queries=20, weight_acq=1.0, weight_fes=0.0, title=""):
    cfg = BQConfig(
        kernel_type="Matern32",
        lengthscale=np.array([0.75, 0.75]),
        noise=1e-6,
        variance=1.0,
        acq_function=acq_function,
        grid_size_2d=(40, 40),
        use_mini=True,
        fast_mini=True,
    )
    r = BayesianQuadratureRunner(system, cfg)
    r.initialize(np.array([[-1.507964474, 1.193805208 ],  [0.879645943, -0.879645943]]))
    r.run(n_queries=n_queries, weight_var=weight_acq, weight_fes=weight_fes)
    print(f"\n{title}")
    r.plot_fes(show=True)
    r.plot_acq(show=True, full=True)
    return r

In [ ]:
r_ivr = run_buq_acq("IVR",title="IVR")

In [ ]:
r_us = run_buq_acq("US", title="Uncertainty Sampling")

**Questions:**
- Do US and sample in different locations than IVR?
- Which gives the best FES reconstruction after the same number queries?
- Why is IVR the most natural choice for free energy estimation?

## Summary

| | Notebook 1: BO | Notebook 2: BUQ |
|---|---|---|
| **Objective** | Find zero of force | Reconstruct full FES |
| **GP models** | \|force(phi, psi)\| | force(phi, psi) directly |
| **Acquisition** | EI / PI / LCB | IVR / US  |
| **Sampling** | Clusters near minimum | Spreads across domain |
| **Output** | Best (phi, psi) found | F(phi, psi) with error bars |

BUQ is the principled approach when you need the **full free energy surface** and want to know **how uncertain** your estimate is -- which is exactly what
matters in enhanced sampling MD.

You have reached the end of notebook 2! Wrapping up notebook 1 and notebook 2, we can conclude that BO approaches are interesting when you have an unkown function which is expensive to evaluate (experiments or simulations). How you set up your BO depends on your goal, are you interested in the minimum, or in the whole function, or maybe only at points with a high derivative. In other words, your goal defines your acquisition function.